In [29]:

from pathlib import Path
import requests

# Create dataset directory
data_dir = Path("data/tiny_shakespeare")
data_dir.mkdir(parents=True, exist_ok=True)

# Official raw dataset URL
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Download dataset
response = requests.get(url, timeout=30)
response.raise_for_status()

# Save locally
train_file = data_dir / "input.txt"
train_file.write_text(response.text, encoding="utf-8")

# Read dataset
with open(train_file, "r", encoding="utf-8") as f:
    text = f.read()

print("Dataset downloaded successfully!")
print("File:", train_file.resolve())
print("Total characters:", len(text))
print("\nFirst 500 characters:\n")
print(text[:500])


Dataset downloaded successfully!
File: /Users/nitish/Documents/Own LLM/data/tiny_shakespeare/input.txt
Total characters: 1115394

First 500 characters:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [ ]:
# read it in to inspect it
with open('data/tiny_shakespeare/input.txt', 'r', encoding='utf-8') as f:
  text = f.read()
  
# print the length of the characters
print("Length of dataset in characters: ", len(text))
# print first 1000 characters
print(text[:1000])

Length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hung

In [ ]:
# Get all unique characters
chars = sorted(list(set(text)))

# Calculate vocabulary size
vocab_size = len(chars)

# Print all unique characters
print(''.join(chars))

# Print vocabulary size
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [38]:

# String to Integer
stoi = {ch: i for i, ch in enumerate(chars)}

# Integer to String
itos = {i: ch for i, ch in enumerate(chars)}

# Encode: string -> list of integers
encode = lambda s: [stoi[c] for c in s]

# Decode: list of integers -> string
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hi there"))
print(decode(encode("hi there")))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch

data = torch.tensor(encode(chars), dtype=torch.long)
print(data.shape, data.dtype)
print(data[: 1000])  # the 1000 characters we looked at earlier will to the GPT look like this

torch.Size([65]) torch.int64
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64])


In [50]:
# Let's now split up the data into train and validation sets
n = int(0.90 * len(data))
train_data = data[:n]  # first 90% will be train, rest val
val_data = data[n:]

print("Total tokens:", len(data))
print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))

print("Training shape:", train_data.shape)
print("Validation shape:", val_data.shape)

Total tokens: 65
Training tokens: 58
Validation tokens: 7
Training shape: torch.Size([58])
Validation shape: torch.Size([7])


In [ ]:
block_size = 8
train_data[:block_size + 1]

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8])

In [52]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"When input is {context} the target: {target}")

When input is tensor([0]) the target: 1
When input is tensor([0, 1]) the target: 2
When input is tensor([0, 1, 2]) the target: 3
When input is tensor([0, 1, 2, 3]) the target: 4
When input is tensor([0, 1, 2, 3, 4]) the target: 5
When input is tensor([0, 1, 2, 3, 4, 5]) the target: 6
When input is tensor([0, 1, 2, 3, 4, 5, 6]) the target: 7
When input is tensor([0, 1, 2, 3, 4, 5, 6, 7]) the target: 8


In [59]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1: i+block_size+1] for i in ix])
  return x, y

xb, yb = get_batch('train')
print('Inputs: ')
print(xb.shape)
print(xb)
print('Targets: ')
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b, :t+1]
    target = yb[b,t]
    print(f"When input is {context.tolist()} the target: {target}")

Inputs: 
torch.Size([4, 8])
tensor([[15, 16, 17, 18, 19, 20, 21, 22],
        [ 7,  8,  9, 10, 11, 12, 13, 14],
        [42, 43, 44, 45, 46, 47, 48, 49],
        [ 0,  1,  2,  3,  4,  5,  6,  7]])
Targets: 
torch.Size([4, 8])
tensor([[16, 17, 18, 19, 20, 21, 22, 23],
        [ 8,  9, 10, 11, 12, 13, 14, 15],
        [43, 44, 45, 46, 47, 48, 49, 50],
        [ 1,  2,  3,  4,  5,  6,  7,  8]])
-----
When input is [15] the target: 16
When input is [15, 16] the target: 17
When input is [15, 16, 17] the target: 18
When input is [15, 16, 17, 18] the target: 19
When input is [15, 16, 17, 18, 19] the target: 20
When input is [15, 16, 17, 18, 19, 20] the target: 21
When input is [15, 16, 17, 18, 19, 20, 21] the target: 22
When input is [15, 16, 17, 18, 19, 20, 21, 22] the target: 23
When input is [7] the target: 8
When input is [7, 8] the target: 9
When input is [7, 8, 9] the target: 10
When input is [7, 8, 9, 10] the target: 11
When input is [7, 8, 9, 10, 11] the target: 12
When input is [7, 8